In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [22]:
# 1. Load the dataset
# ---------------------------
# Replace 'E-commerce Customer Behavior - Sheet1.csv' with your actual file path
df = pd.read_csv('E-commerce Customer Behavior - Sheet1.csv')

In [23]:
# Remove any rows with missing Satisfaction Level (e.g., rows 172 and 244)
df = df.dropna(subset=['Satisfaction Level'])
# Create a binary 'Satisfied' column: True if 'Satisfied', else False (for Neutral/Unsatisfied)
df['Satisfied'] = df['Satisfaction Level'] == 'Satisfied'

In [24]:
# For counts, we'll use 'Satisfaction Status' = 'Satisfied' or 'Not Satisfied'
df['Satisfaction Status'] = df['Satisfied'].map({True: 'Satisfied', False: 'Not Satisfied'})

In [ ]:
# ---------------------------
# 2. Overall satisfaction pie chart
# ---------------------------
overall_counts = df['Satisfaction Status'].value_counts()
plt.figure(figsize=(7,7))
plt.pie(overall_counts, labels=overall_counts.index, autopct='%1.0f%%', 
        colors=['#66b3ff', '#ff9999'], startangle=90, explode=(0.05, 0))
plt.title('Overall Customer Satisfaction', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------
# 3. Satisfaction by gender (side-by-side bar chart)
# ---------------------------
# Create a cross-tabulation of counts
gender_status = pd.crosstab(df['Gender'], df['Satisfaction Status'])
gender_status.plot(kind='bar', figsize=(8,6), color=['#66b3ff', '#ff9999'], edgecolor='black')
plt.title('Satisfaction by Gender', fontsize=14)
plt.xlabel('Gender')
plt.ylabel('Number of Customers')
plt.xticks(rotation=0)
plt.legend(title='Satisfaction Status')
for container in plt.gca().containers:
    plt.gca().bar_label(container, label_type='edge')
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------
# 4. Satisfaction by membership type (stacked bar chart)
# ---------------------------
membership_status = pd.crosstab(df['Membership Type'], df['Satisfaction Status'])
membership_status.plot(kind='bar', stacked=True, figsize=(8,6), 
                       color=['#66b3ff', '#ff9999'], edgecolor='black')
plt.title('Satisfaction by Membership Type', fontsize=14)
plt.xlabel('Membership Type')
plt.ylabel('Number of Customers')
plt.xticks(rotation=0)
plt.legend(title='Satisfaction Status')
for container in plt.gca().containers:
    plt.gca().bar_label(container, label_type='center')
plt.tight_layout()
plt.show()

In [28]:
# ---------------------------
# 5. Satisfaction by membership and gender (grouped bar with hatching)
# ---------------------------
# Create a pivot table: rows = Membership, columns = Gender, values = Satisfaction Status counts
# We'll make two subplots? Actually grouped by membership, with bars for each gender and status.
# Better: Use seaborn's catplot or a manual grouped bar with hatches.

# Prepare data: counts for each combination
grouped = df.groupby(['Membership Type', 'Gender', 'Satisfaction Status']).size().reset_index(name='Count')
# Pivot for easier plotting
pivot = grouped.pivot(index=['Membership Type', 'Gender'], columns='Satisfaction Status', values='Count').fillna(0)
pivot = pivot.reset_index()

In [ ]:
# Plot grouped bars: for each membership, two groups (Male/Female), each with two bars (Satisfied/Not Satisfied)
memberships = pivot['Membership Type'].unique()
genders = ['Male', 'Female']
statuses = ['Satisfied', 'Not Satisfied']
x = np.arange(len(memberships))  # label locations
width = 0.35  # width of each bar group

fig, ax = plt.subplots(figsize=(10,6))

# Colors and hatches
colors = {'Satisfied': '#66b3ff', 'Not Satisfied': '#ff9999'}
hatches = {'Male': '///', 'Female': '\\\\\\'}  # different hatching per gender
# For each gender, plot a set of bars
for i, gender in enumerate(genders):
    # Extract counts for this gender across memberships
    sat_counts = []
    not_sat_counts = []
    for mem in memberships:
        row = pivot[(pivot['Membership Type'] == mem) & (pivot['Gender'] == gender)]
        sat_counts.append(row['Satisfied'].values[0] if not row.empty else 0)
        not_sat_counts.append(row['Not Satisfied'].values[0] if not row.empty else 0)
          # Position for this gender's bars
    offset = (i - 0.5) * width
    # Plot Satisfied bars
    bars1 = ax.bar(x + offset, sat_counts, width, label=f'{gender} - Satisfied', 
                   color=colors['Satisfied'], edgecolor='black', hatch=hatches[gender])
    # Plot Not Satisfied bars on top (stacked within each gender group)
    bars2 = ax.bar(x + offset, not_sat_counts, width, bottom=sat_counts,
                   label=f'{gender} - Not Satisfied', color=colors['Not Satisfied'], 
                   edgecolor='black', hatch=hatches[gender])
# Add labels and title
ax.set_xlabel('Membership Type', fontsize=12)
ax.set_ylabel('Number of Customers', fontsize=12)
ax.set_title('Satisfaction by Membership Type and Gender', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(memberships)
ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1))
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------
# Bonus: Print the numbers to verify
# ---------------------------
print("Overall counts:\n", overall_counts)
print("\nBy gender:\n", gender_status)
print("\nBy membership:\n", membership_status)
print("\nDetailed by membership & gender:\n", grouped)